In [1]:
%load_ext autoreload
%autoreload 2

This notebook demonstrates how to use the clemcore OpenEnv integration to run a
single-player game with the game **"Wordle"**, and how to plug in a self-hosted
OpenAI-compatible model server as an agent backend.

You will learn how to:

1. Set up the environment and install dependencies.
2. Register and configure a model as an agent backend.
3. Implement a simple custom describer agent.
4. Run an automated gameplay session and inspect the interactions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/phisad/playpen/blob/main/examples/openenv/wordle.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"wordle"`, but this pattern works for any 1‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [1]:
import os

# Specify the game name here (this code can be adapted to any 1-player game)
GAME_NAME = "wordle"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `wordle` listed by `clem list games -s wordle`.

If you do **not** see `wordle`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [ ]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

In [ ]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

# 2. Register and configure a model backend

We will use a self-hosted OpenAI-compatible model server and register the model
under the name `"clp-chat"`.

This name (`"clp-chat"`) will later be used when we assign the model to a player
in the Gymnasium environment.

In [ ]:
# Register the remote model under the name "clp-chat"
!clem register model -n "clp-chat" -v "backend=openai_compatible,model_id=Qwen/Qwen3-VL-30B-A3B-Thinking-FP8" --cwd

You can achieve the same registration programmatically if you prefer not to use
the CLI:

In [ ]:
from clemcore.backends import ModelRegistry

registry = ModelRegistry.register("clp-chat", backend="openai_compatible",
                                  model_id="Qwen/Qwen3-VL-30B-A3B-Thinking-FP8")
registry.get_first_model_spec_that_unify_with("clp-chat")

## 2.2. Register API key and base URL

To connect to your OpenAI-compatible model server, we also register:

- An API key
- An organisation (if applicable)
- A base URL for the endpoint

Replace the placeholders with your actual values.

In [ ]:
API_KEY = "<insert-api-key-here>"
ORGANIZATION = "<insert-organisation-here>"
BASE_URL = "<insert-base-url-here>"

In [ ]:
# Register the credentials and base URL for the "openai_compatible" backend
!clem register key -n "openai_compatible" -v "api_key=$API_KEY,organisation=$ORGANIZATION,base_url=$BASE_URL" --cwd

Programmatic equivalent:

In [ ]:
from clemcore.backends import KeyRegistry

KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

# 3. Implement a custom guesser agent

Next, we implement a simple custom guesser agent by subclassing `ClemAgent`.
The agent:

- Uses the `"clp-chat"` model backend.
- Generates a guess based on the full interaction history in the Wordle game.

This is just a toy example to show how you can plug in your own logic.

In [2]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model


class MyAgenticGuesser(ClemAgent):
    """
    Simple example agent for Wordle.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        # with a reasoning model we must allow more tokens to account for the thinking process
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.7, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe our own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text


guesser = MyAgenticGuesser()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'

2026-01-12 18:24:04,987 - clemcore.backends - INFO - Found registered model spec that unifies 

# 4. Run an automated gameplay session

We now:

1. Set up an OpenEnv environment for the game where we use the registered `"clp-chat"` model to play `player_1` as part of the environment.
2. Run through a single episode and log the interaction at each step where we use our custom describer to play `player_0` as the learning agent.

Note that:
- Setting `--single_pass` allows to iterate through all game instances only once times, e.g., for evaluation.
- You can also restrict which instances are used via the `--split [train,validation]` option.

## 4.1. Starting the server and connecting the client

Open a terminal in the notebook folder and run the `clem serve` command to start the OpenEnv environment:

```bash
clem serve --game wordle
```

The output should look similar to:
```
INFO:     Started server process [73210]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

From this log you obtain the host and port (here `http://0.0.0.0:8000`) that the client should connect to.

In [3]:
from clemcore.clemgame import ClemGameEnv

game_env = ClemGameEnv(base_url="http://0.0.0.0:8000")

## 4.2. Run through an episode and log the interaction

In [4]:
from playpen.agents.openenv import ClemGameEnvAgent

# Wrap the agent in a ClemGameEnvAgent to use it with the OpenEnv client
openenv_guesser = ClemGameEnvAgent(guesser)

last_obs = game_env.reset()
context_response_pairs: list[tuple] = []
counter = 0
while not last_obs.done:
    print(f"Round: {counter} ...")
    action = openenv_guesser(last_obs)  # this is a StepResult[ClemGameObservation]
    obs = game_env.step(action)
    context_response_pairs.append((last_obs.observation, action.response, last_obs.reward or 0.))
    last_obs = obs
    counter += 1

# Reset agent state between episodes (if it maintains any internal memory)
openenv_guesser.reset()

print(f"Episode took these {len(context_response_pairs)} steps:")
print("-" * 20)
for idx, (context, response, reward) in enumerate(context_response_pairs):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"Describer <- Context:", context)
    print(f"Describer -> Response:", response)
    print("-" * 20)

print("Final reward:", reward)

Round: 0 ...
Round: 1 ...
Round: 2 ...
Round: 3 ...
Round: 4 ...
Round: 5 ...
Episode took these 6 steps:
--------------------
Step 0 / Reward 0.00:
Describer <- Context: ClemGameObservation(done=False, reward=None, metadata={}, context={'role': 'user', 'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be usefu